# 01 · 데이터 이해 · 전처리 · Feature Engineering

MLOps 파이프라인 1단계. 본 노트북은 **요구사항 #1(데이터 이해) + #4(전처리/Feature Engineering)** 를 담당한다.

- 원천 데이터: `data/books.csv` (베스트셀러 메타데이터)
- 평가 데이터: `data/golden_set.json` (30 케이스 골든셋)
- 산출물: `data/books_clean.csv` (정제본, DVC로 버전 관리) + `data/data_registry.json` (sha256 매니페스트)

> 재사용 가능한 모든 전처리/피처 함수는 `src/data_prep.py`에 모듈로 분리되어
> 02(실험)·04(XAI) 노트북과 백엔드(DB 시드)가 **동일 로직**을 공유한다.

In [ ]:
import sys, os, json
sys.path.append('..')
import pandas as pd
import matplotlib.pyplot as plt
from src import data_prep as dp
pd.set_option('display.max_colwidth', 40)
print('src/data_prep 로드 완료')

## 1. 데이터 이해 (EDA)

원천 CSV는 summary 필드에 비인용 콤마가 섞여 표준 파서가 깨지는 **데이터 품질 결함**이 있다.
`read_books_robust()`로 견고하게 로드한다(요구사항 #1: 데이터의 실제 상태를 직시).

In [ ]:
raw = dp.read_books_robust('../data/books.csv')
print('shape:', raw.shape)
print('\n결측치:')
print(raw.replace('', pd.NA).isna().sum())
raw.head(3)

### 1.1 중복 점검 — 동일 도서의 표기 흔들림

'총 균 쇠' / '총균쇠'처럼 같은 책이 공백·표기 차이로 중복 적재되어 있다.

In [ ]:
raw2 = dp.add_book_features(raw)
dup_keys = raw2['title_norm'].value_counts()
print('정규화 제목 기준 중복 그룹:')
print(dup_keys[dup_keys > 1])
print('\n중복 상세:')
raw2[raw2['title_norm'].isin(dup_keys[dup_keys>1].index)][['rank','title','author']]

### 1.2 분포 — 요약 길이 · 소설/비소설

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
raw2['summary_word_len'].plot(kind='hist', bins=10, ax=ax[0], color='#1F4E79', edgecolor='white')
ax[0].set_title('Summary word length'); ax[0].set_xlabel('words')
raw2['is_likely_nonfiction'].value_counts().sort_index().plot(
    kind='bar', ax=ax[1], color=['#1F4E79','#C0504D'])
ax[1].set_title('0=fiction  1=nonfiction(heuristic)'); ax[1].set_xticklabels(['fiction','nonfiction'], rotation=0)
plt.tight_layout(); plt.show()
print('요약 길이 통계(words):'); print(raw2['summary_word_len'].describe().round(1))

## 2. 전처리 + Feature Engineering

`clean_books()`: ① 제목 정규화 후 중복 제거(대표 rank 1건 유지), ② 저자 표기 보정,
③ 결측 요약 처리, ④ 파생 피처 생성. 결과를 `books_clean.csv`로 저장한다.

In [ ]:
clean = dp.run_books_pipeline('../data/books.csv', '../data/books_clean.csv')
print('정제 후 shape:', clean.shape)
print('파생 피처 컬럼:', [c for c in clean.columns if c not in ['rank','title','author','summary','yes24_url','image_url']])
clean[['title','author','summary_char_len','summary_word_len','summary_sentence_len','is_likely_nonfiction']].head(8)

### 2.1 저자 표기 보정 확인 — '무라카키'→'무라카미'

In [ ]:
print('원천 1Q84 저자:', raw[raw['title']=='1Q84']['author'].values)
print('정제 1Q84 저자:', clean[clean['title']=='1Q84']['author'].values)

## 3. 코치 응답 Feature Engineering (질문 품질의 설명변수)

books 메타 외에, **코치 응답**을 수치 피처로 바꾸는 `featurize_response()`도 제공한다.
이 피처들은 02(실험 비교)와 04(XAI)에서 'Depth Score를 무엇이 끌어올리는가'를 설명하는 데 쓰인다.

| 피처 | 의미 |
|------|------|
| resp_word_len | 응답 길이(단어) |
| n_question_marks | 물음표 수(질문성) |
| ends_with_question | 물음표로 끝나는가(열린 질문 경향) |
| user_overlap | 사용자 발화와 토큰 Jaccard(사용자 그라운딩) |
| second_person_refs | '방금/당신/말씀…' 사용자 지시 표현 |
| has_quote_mark | 인용부호 포함(지어낸 인용 위험 신호) |

In [ ]:
samples = [
    {'user_msg':'"새는 알을 깨고 나온다"는 구절이 마음에 박혔어요.',
     'assistant_msg':'방금 말씀하신 그 구절에서, 당신이 깨고 싶은 "익숙한 세계"는 무엇인가요?'},
    {'user_msg':'윈스턴의 결말이 패배주의 같았어요.',
     'assistant_msg':'그렇군요. 좋은 작품이죠.'},
]
demo = dp.featurize_frame(pd.DataFrame(samples))
demo[['resp_word_len','n_question_marks','ends_with_question','user_overlap','second_person_refs','has_quote_mark']]

## 4. 골든셋 로드 + 데이터 버전 매니페스트

평가용 골든셋(30 케이스)은 `data/golden_set.json`으로 외부화되어 DVC로 버전 관리된다.
전처리 산출물의 sha256을 `data_registry.json`에 기록한다(DVC 보조 · 사람이 읽는 버전 기록).

In [ ]:
gs = json.load(open('../data/golden_set.json', encoding='utf-8'))
print('golden set version:', gs['version'], '| cases:', gs['n_cases'], '| scenarios:', gs['scenarios'])

entry = dp.write_manifest(
    ['../data/books.csv', '../data/books_clean.csv', '../data/golden_set.json'],
    version='v1.0', manifest_path='../data/data_registry.json',
    note='01 노트북 산출: 정제본 + 골든셋')
print('\n데이터 매니페스트(v1.0):')
print(json.dumps(entry, ensure_ascii=False, indent=2))

## 5. 다음 단계

- 정제본 `books_clean.csv`는 백엔드 DB 시드와 02/04 노트북이 공유한다.
- **DVC**로 `data/`를 버전 관리한다: `dvc add data/books.csv data/books_clean.csv data/golden_set.json` → `.dvc` 메타가 Git에 커밋된다(`data/README.md` 참고).
- 다음: `02_model_experiment_mlflow.ipynb` (모델×프롬프트 실험 추적·비교), `03_evaluation_calibration.ipynb`(평가+사람보정), `04_xai.ipynb`(해석).